In [2]:
from ray.tune.analysis     import ExperimentAnalysis
# from   ray.tune.schedulers import ASHAScheduler
# import ray.cloudpickle     as pickle
# from   ray import tune
# from   ray import train
# from ray.train import Checkpoint, get_checkpoint
from train import GenerateModel, federate_model
from torch.utils.data import DataLoader, TensorDataset
from metrics import eval_model, compare_metric
from functools import partial
from tqdm import tqdm
import json
import torch
import os

import datetime

working_dir  = "/home/drew/FL-with-MIMIC/Replicating Mullenbach/AWS" 

In [3]:
def save_json(data,filepath): 
    with open(filepath, mode = 'w+') as f:
        json.dump(data,fp = f)

def load_json(filepath):
    with open(filepath, mode = 'r') as f:
        data = json.load(f)
    return data

In [4]:
def load_data(type):
    working_dir  = "/home/drew/FL-with-MIMIC/Replicating Mullenbach/AWS" 
    X_val  = torch.load(os.path.join(working_dir,"Data",f"X_{type}.pt"))
    Y_val  = torch.load(os.path.join(working_dir,"Data",f"Y_{type}.pt"))
    return DataLoader(TensorDataset(X_val,Y_val),batch_size=32,shuffle=False)

def log_detail(desc: str, file: str):
    with open(file, mode = 'a') as f:
        time_now = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        f.write(f'{time_now}: {desc}\n')

In [5]:
logging_path = os.path.join(working_dir,"S3_Bucket", "logs","logger.log")
low_ge_path  = os.path.join(working_dir,"S3_Bucket", "logs","low_ge.pt")
max_auc_path = os.path.join(working_dir,"S3_Bucket", "logs","max_auc.pt")
model_path   = os.path.join(working_dir,"S3_Bucket")
history_metrics_path = os.path.join(working_dir,"S3_Bucket", "logs","metric_history.json")
latest_model_path    = os.path.join(working_dir,"S3_Bucket", "logs","latest_model.pt")
os.path.isfile(logging_path)

True

In [6]:
######################################## CONFIG
config = {
        "batch_size" : 32,
        "lr"         : 0.00001,
        "n_filters"  : 21,
        "window_size": 6,
        "epochs"     : 2,
        "rounds"     : 1
    }
######################################## Global Model
model = GenerateModel(table_path   = os.path.join(working_dir,"Model","processed_full.w2v"),
                    num_of_filters = config['n_filters'],
                    kernel_size    = config['window_size'])

model_path = '/home/drew/FL-with-MIMIC/Replicating Mullenbach/AWS/S3_Bucket/logs/latest_model.pt'
state_dict = torch.load(f = model_path,map_location=torch.device("cpu"), weights_only=True)
model.load_state_dict(state_dict)


<All keys matched successfully>

In [ ]:
########################################
min_error     = float('inf')
max_auc_macro = 0
metric_history = []

########################################
for round in tqdm(range(20_000), colour = 'blue', desc = 'Federated Training'):
    trained_model = federate_model(config, model_param = model.state_dict())
    model.load_state_dict(trained_model)
    ######################################## Generalization Error
    train_hist = eval_model(
               model       = model,
               device      = torch.device("cuda"),
               data_loader = load_data(type = 'train')
    )
    val_hist = eval_model(
               model       = model,
               device      = torch.device("cuda"),
               data_loader = load_data(type = 'val')
    )
    metric_history.append((train_hist,val_hist))
    save_json(data = metric_history, filepath=history_metrics_path)
    ######################################## Save the best if it exists
    ge_error = abs(train_hist['auc_macro'] - val_hist['auc_macro'])
    if min_error > ge_error:
        log_detail(desc = f'Min ge_error: {min_error:,.4f}->{ge_error:,.4f};{round}',file = logging_path)
        min_error = ge_error
        torch.save(model.state_dict(), f = low_ge_path)
    
    if val_hist['auc_macro'] > max_auc_macro:
        log_detail(desc = f'max auc_macro: {max_auc_macro:,.4f}->{val_hist['auc_macro']:,.4f};{round}',file = logging_path)
        max_auc_macro = val_hist['auc_macro']
        torch.save(model.state_dict(), f = max_auc_path)
    ######################################## Save the best if it exists
    torch.save(model.state_dict(), f = latest_model_path)

____